# Avaliação Prática 1 — execução em LOTE, sem supervisão

Este notebook não é para rodar célula a célula. É para **`Save Version → Save & Run All (Commit)`**:
o Kaggle o executa nos servidores dele, **com o navegador fechado**, e ao terminar ele
**envia os resultados direto para o GitHub**. Nada precisa ser baixado, e nada se perde se a
sessão for reciclada depois do push.

As execuções já concluídas (as 5 estratégias e as investigações 2a e 4a — 15 artefatos)
**estão versionadas no repositório** e são recuperadas pelo `git clone`. Este lote roda
apenas o que falta: as ablações 4(b) e 4(c), uma em cada GPU.

---

## Antes de commitar a versão

**1. `Settings → Accelerator → GPU T4 x2`** — o `x2` importa; sem ele não há paralelismo.

**2. `Settings → Internet → On`**.

**3. `Add-ons → Secrets` → adicione `GITHUB_TOKEN`** — um Personal Access Token do GitHub
com escopo `repo` ([criar aqui](https://github.com/settings/tokens/new?scopes=repo)) — e
**anexe-o a este notebook** (botão `Attach to notebook`).

Sem o token, o lote treina tudo e **não consegue entregar** — a última célula falha, e as
~2 h de GPU se perdem. Por isso a primeira célula verifica o token **antes** de treinar
qualquer coisa.

Duração estimada: **~2 h**.

In [ ]:
# 1. VERIFICAR TUDO ANTES DE GASTAR GPU.
#    Cada falha abaixo é silenciosa e cara: sem GPU o lote levaria dias; com transformers 5.x
#    o ViT treinaria com pesos aleatórios; sem token, o resultado seria treinado e perdido.
import os, pathlib, subprocess

from kaggle_secrets import UserSecretsClient
os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert os.environ['GITHUB_TOKEN'], 'GITHUB_TOKEN vazio'
print('token: OK (não será impresso nem gravado)')

REPO = pathlib.Path('/kaggle/working/machine-learning-fundamentals')
ACTIVITY = REPO / 'activities' / 'avaliacao-pratica-1'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fsd-dantas/machine-learning-fundamentals.git',
                    str(REPO)], check=True)
%cd {ACTIVITY}
!pip install -q -r requirements.txt

import torch, transformers, tensorflow as tf
assert torch.cuda.is_available(), 'Sem GPU: Settings → Accelerator → GPU T4 x2'
assert transformers.__version__ < '5', "transformers 5.x quebra o ViT"
print('GPUs visíveis:', len(tf.config.list_physical_devices('GPU')))
print('artefatos recuperados do repositório:',
      len(list((ACTIVITY / 'results').glob('*.json'))))

In [ ]:
# 2. O que falta rodar.
!python status.py

In [ ]:
# 3. AS DUAS ABLAÇÕES QUE FALTAM — uma em cada GPU (~2 h).
#    4(c) política: lecture, flip, flip_crop, strong × 2 seeds
#    4(b) otimizador: Adam, AdamW, SGD+Nesterov, RMSprop × 2 seeds
!python run_parallel.py \
  "s4_augmentation.py --ablation-policy --seeds 42 7" \
  "s4_augmentation.py --ablation-optimizer --seeds 42 7"
!python report.py

In [ ]:
# 4. ENTREGAR — push direto para o GitHub. Sem download, sem espera.
#    Roda mesmo que a célula anterior tenha falhado parcialmente: o que existir em
#    results/ é evidência, e evidência parcial vale mais que evidência perdida.
!python push_results.py --branch results/ap1-ablations

# rede de segurança: o zip também fica na aba Output desta versão
%run -i baixar.py